# 02 — CICIDS2017 Dataset Summary

This notebook provides a thorough summary of the CICIDS2017 dataset after ingestion and preprocessing.

**Goals:**
- Document the processed dataset's scale, schema, and label distribution
- Validate the temporal train / validation / test split
- Highlight class imbalance and its implications for model training
- Compare BENIGN vs ATTACK flow characteristics

**Prerequisites:** run `ingest.py` and `preprocess.py` first (or `python -m src.run_pipeline --only preprocess`).

## 0. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROCESSED = "../data/processed"

PATHS = {
    "combined" : os.path.join(PROCESSED, "cicids2017.parquet"),
    "train"    : os.path.join(PROCESSED, "cicids_train.parquet"),
    "valid"    : os.path.join(PROCESSED, "cicids_valid.parquet"),
    "test"     : os.path.join(PROCESSED, "cicids_test.parquet"),
}

missing = [k for k, v in PATHS.items() if not os.path.exists(v)]
if missing:
    print(f"Missing files: {missing}")
    print("Run: python -m src.run_pipeline --only preprocess")
else:
    print("All required files found.")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Dataset scale

In [ ]:
combined = pd.read_parquet(PATHS["combined"])
train    = pd.read_parquet(PATHS["train"])
valid    = pd.read_parquet(PATHS["valid"])
test     = pd.read_parquet(PATHS["test"])

total = len(combined)

print("=" * 50)
print("CICIDS2017 — Combined Dataset")
print("=" * 50)
print(f"Rows    : {len(combined):>10,}")
print(f"Columns : {combined.shape[1]:>10}")
print(f"Memory  : {combined.memory_usage(deep=True).sum()/1e6:>9.1f} MB")

print("\n" + "=" * 50)
print("Temporal Split")
print("=" * 50)
for name, df in [("Train", train), ("Validation", valid), ("Test", test)]:
    attacks = (df["Label"] != "BENIGN").sum()
    print(f"{name:<12}: {len(df):>9,} rows ({len(df)/total*100:.1f}%)  "
          f"| Attacks: {attacks:>7,} ({attacks/len(df)*100:.1f}%)")

## 2. Label distribution — full dataset

In [ ]:
counts = combined["Label"].value_counts()
pct    = combined["Label"].value_counts(normalize=True).mul(100).round(3)

label_df = pd.DataFrame({"Count": counts, "Pct %": pct})
print("Full dataset label distribution:\n")
print(label_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CICIDS2017 — Label Distribution", fontsize=13, fontweight="bold")

# Left: all classes, log scale
ax = axes[0]
colors = ["#2196F3" if l == "BENIGN" else "#F44336" for l in counts.index]
counts.plot(kind="bar", ax=ax, color=colors, edgecolor="none")
ax.set_yscale("log")
ax.set_title("All Classes (log scale)")
ax.set_ylabel("Flow count")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Right: attack classes only
ax2 = axes[1]
attack_counts = counts.drop("BENIGN")
attack_counts.plot(kind="bar", ax=ax2, color="#F44336", edgecolor="none")
ax2.set_title("Attack Classes Only")
ax2.set_ylabel("Flow count")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=35)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.show()

## 3. Temporal split — label distribution per split

We split by source file (day of week) rather than random shuffle. This is the correct approach for time-series network data — it prevents data leakage and mimics real deployment conditions.

In [ ]:
for name, df in [("Train (Mon–Wed)", train), ("Validation (Thu)", valid), ("Test (Fri)", test)]:
    print(f"\n{'─'*45}")
    print(f"  {name}")
    print(f"{'─'*45}")
    vc = df["Label"].value_counts()
    pct = df["Label"].value_counts(normalize=True).mul(100).round(2)
    print(pd.DataFrame({"Count": vc, "Pct %": pct}).to_string())

In [ ]:
# Stacked bar: attack vs benign ratio per split
splits = {"Train": train, "Validation": valid, "Test": test}
split_names = list(splits.keys())
benign_pcts  = [len(df[df["Label"] == "BENIGN"]) / len(df) * 100 for df in splits.values()]
attack_pcts  = [100 - b for b in benign_pcts]

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(split_names, benign_pcts, label="BENIGN", color="#2196F3")
bars2 = ax.bar(split_names, attack_pcts, bottom=benign_pcts, label="ATTACK", color="#F44336")

for bar, pct in zip(bars2, attack_pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + bar.get_height()/2,
            f"{pct:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=10)

ax.set_ylabel("Percentage of flows")
ax.set_title("BENIGN vs ATTACK ratio per split")
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 4. Feature overview

CICIDS2017 has 78 numeric features extracted by CICFlowMeter from raw PCAPs.

In [ ]:
numeric = combined.select_dtypes(include=[np.number])
print(f"Numeric features : {numeric.shape[1]}")
print(f"Non-numeric cols : {combined.select_dtypes(exclude=[np.number]).columns.tolist()}")

In [ ]:
# Top 10 features by variance
top10 = numeric.var().nlargest(10)
print("Top 10 features by variance (proxy for discriminative power):")
print(top10.to_string())

## 5. BENIGN vs ATTACK feature contrast

Which features differ most between normal and attack traffic? This motivates the features our models will learn from.

In [ ]:
benign_mean = numeric[combined["Label"] == "BENIGN"].mean()
attack_mean = numeric[combined["Label"] != "BENIGN"].mean()

diff = (attack_mean - benign_mean).abs().nlargest(12)

contrast = pd.DataFrame({
    "BENIGN mean" : benign_mean[diff.index].round(2),
    "ATTACK mean" : attack_mean[diff.index].round(2),
    "|difference|" : diff.round(2),
})

print("Top 12 features by BENIGN–ATTACK mean difference:")
print(contrast.to_string())

In [ ]:
top5 = diff.head(5).index.tolist()
fig, axes = plt.subplots(1, len(top5), figsize=(16, 4))
fig.suptitle("Distribution: BENIGN vs ATTACK — top 5 differentiating features", fontsize=11)

for ax, col in zip(axes, top5):
    benign_vals = combined[combined["Label"] == "BENIGN"][col].replace([np.inf, -np.inf], np.nan).dropna()
    attack_vals = combined[combined["Label"] != "BENIGN"][col].replace([np.inf, -np.inf], np.nan).dropna()
    
    # Clip to 99th percentile to avoid outlier-dominated bins
    upper = max(benign_vals.quantile(0.99), attack_vals.quantile(0.99))
    
    ax.hist(benign_vals.clip(upper=upper), bins=40, alpha=0.6, color="#2196F3", label="BENIGN", density=True)
    ax.hist(attack_vals.clip(upper=upper), bins=40, alpha=0.6, color="#F44336", label="ATTACK", density=True)
    ax.set_title(col[:30], fontsize=8)
    ax.set_xlabel("")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 6. Class imbalance — implications for modelling

CICIDS2017 is heavily imbalanced: ~80% BENIGN, some attack classes have <50 samples. Our pipeline addresses this by:
- `train_model.py` — balanced sampling (50k BENIGN, 50k ATTACK) for binary classification
- `train_multiclass_model.py` — per-class sampling (up to 20k per class) with `class_weight='balanced_subsample'`
- `train_anomaly_model.py` — trains only on BENIGN flows (unsupervised; imbalance is irrelevant)

In [ ]:
attack_counts_only = combined[combined["Label"] != "BENIGN"]["Label"].value_counts()
benign_count = (combined["Label"] == "BENIGN").sum()

print(f"BENIGN flows      : {benign_count:>9,}")
print(f"All attack flows  : {attack_counts_only.sum():>9,}")
print(f"\nRarest attack classes (< 100 samples):")
rare = attack_counts_only[attack_counts_only < 100]
print(rare.to_string() if not rare.empty else "  None")
print(f"\n→ These rare classes benefit most from class weighting and balanced sampling.")

## Summary

| Property | Value |
|---|---|
| Dataset | CICIDS2017 |
| Total flows | ~2.83 million |
| Features | 78 numeric |
| Attack classes | 14 |
| Class imbalance | ~80% BENIGN |
| Split strategy | Temporal by source file (no data leakage) |
| Train period | Mon – Wed |
| Validation period | Thu |
| Test period | Fri |

**Next:** `03_cicids_temporal_split.ipynb` — deeper look at the temporal split and feature engineering.